<a href="https://colab.research.google.com/github/thehmfpk/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check
**Lane:** same as ML-04/ML-07 — `fact_content_daily_performance`, month `2026-03`, decision moment = day 15. Target this feature vector serves: `is_declining_label` (clicks fall days 16-31 vs days 1-15), same as ML-04.

Load `building-baselines`-style rigor + `flyrank/flyrank-data` per `skills/README.md` if working with an assistant.


## 0. Setup — load March 2026 partition directly

In [1]:
import pandas as pd, numpy as np, os
from datasets import load_dataset
from huggingface_hub import HfApi
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files('FlyRank/internship-warehouse', repo_type='dataset')
march_files = [f for f in all_files if 'fact_content_daily_performance' in f
               and 'sample' not in f and '2026-03' in f]
if not march_files:
    print('[WARNING] No files matched - inspect all_files and fix the filter:')
    for f in [x for x in all_files if 'fact_content_daily_performance' in x and 'sample' not in x][:20]:
        print(' ', f)
    raise ValueError('Fix march_files filter above, then re-run.')

dataset = load_dataset('FlyRank/internship-warehouse', data_files={'train': march_files}, split='train')
needed_cols = ['report_date','client_hash_id','content_hash_id',
               'gsc_data_available','gsc_impressions','gsc_clicks','gsc_avg_position']
cols_present = [c for c in needed_cols if c in dataset.column_names]
dataset = dataset.select_columns(cols_present)
raw = dataset.to_pandas()
raw['report_date'] = pd.to_datetime(raw['report_date'])
raw['day_of_month'] = raw['report_date'].dt.day
raw = raw[raw['gsc_data_available'] == True].copy()
print(f'Loaded {len(raw)} GSC-available rows for March 2026.')


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 3611061 GSC-available rows for March 2026.


## 1. Build the feature vector

All features computed from **days 1-15 only** (the decision moment). Label uses days 16-31, kept completely separate and never joined into the feature-building step until section 3's leakage test.

In [2]:
first_half = raw[raw['day_of_month'] <= 15]
second_half = raw[raw['day_of_month'] > 15]

# --- Engineered numeric features (days 1-15 only) ---
feat = (first_half.groupby(['client_hash_id','content_hash_id'])
        .agg(clicks_first_half=('gsc_clicks','sum'),
             impressions_first_half=('gsc_impressions','sum'),
             days_with_position_fh=('gsc_avg_position', lambda s: (s > 0).sum()),
             avg_position_first_half=('gsc_avg_position', lambda s: s[s > 0].mean()),
             active_days_first_half=('gsc_impressions', lambda s: (s > 0).sum()))
        .reset_index())

feat = feat[feat['impressions_first_half'] > 0].copy()  # need impressions to compute a CTR at all

# --- Derived numeric feature ---
feat['ctr_first_half'] = feat['clicks_first_half'] / feat['impressions_first_half']

# --- Missing-value handling: avg_position can be NaN if zero days had real position data ---
feat['has_position_data'] = feat['avg_position_first_half'].notna().astype(int)
feat['avg_position_first_half'] = feat['avg_position_first_half'].fillna(feat['avg_position_first_half'].median())
# NOTE: filled with the median + flagged, per flyrank-data's warning against a blind fillna(0)
# injecting a false 'great position' signal - 0 would look like rank 1, which is wrong.

# --- Categorical feature 1: position bucket (one-hot) ---
position_bins = [0, 3, 6, 10, 20, 50, np.inf]
position_labels = ['1-3','4-6','7-10','11-20','21-50','51+']
feat['position_bucket_fh'] = pd.cut(feat['avg_position_first_half'], bins=position_bins, labels=position_labels)
position_dummies = pd.get_dummies(feat['position_bucket_fh'], prefix='pos_bucket')

# --- Categorical feature 2: volume tier (one-hot) ---
vol_bins = feat['impressions_first_half'].quantile([0, .25, .5, .75, 1.0]).values
vol_bins[0] = -1
feat['volume_tier_fh'] = pd.cut(feat['impressions_first_half'], bins=vol_bins,
                                 labels=['Q1_low','Q2','Q3','Q4_high'], duplicates='drop')
volume_dummies = pd.get_dummies(feat['volume_tier_fh'], prefix='volume_tier')

feature_vector = pd.concat([
    feat[['client_hash_id','content_hash_id','clicks_first_half','impressions_first_half',
          'ctr_first_half','avg_position_first_half','has_position_data','active_days_first_half']],
    position_dummies, volume_dummies
], axis=1)

print(f'Feature vector: {len(feature_vector)} rows, {feature_vector.shape[1]} columns.')
feature_vector.head()


Feature vector: 151981 rows, 18 columns.


,client_hash_id,content_hash_id,clicks_first_half,impressions_first_half,ctr_first_half,avg_position_first_half,has_position_data,active_days_first_half,pos_bucket_1-3,pos_bucket_4-6,pos_bucket_7-10,pos_bucket_11-20,pos_bucket_21-50,pos_bucket_51+,volume_tier_Q1_low,volume_tier_Q2,volume_tier_Q3,volume_tier_Q4_high
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,1,119,0.008403,12.639599,1,15,False,False,False,True,False,False,False,False,True,False
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,0,33,0.000000,9.225529,1,6,False,False,True,False,False,False,False,True,False,False
2,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,0,72,0.000000,8.094074,1,15,False,False,True,False,False,False,False,True,False,False
3,client_0797ff3a1fc9a6a5,content_1207efddce873942,0,283,0.000000,12.587226,1,15,False,False,False,True,False,False,False,False,True,False
4,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,0,9,0.000000,11.500000,1,5,False,False,False,True,False,False,True,False,False,False


## 2. Feature notes

| Feature | Meaning | Missing handling | Categorical? | Available at decision moment (day 15)? |
|---|---|---|---|---|
| `clicks_first_half` | total GSC clicks, days 1-15 | none needed (0 is a valid real count) | no | yes |
| `impressions_first_half` | total GSC impressions, days 1-15 | rows with 0 impressions dropped (can't compute CTR) | no | yes |
| `ctr_first_half` | clicks/impressions, days 1-15 | derived, undefined only when impressions=0 (already dropped) | no | yes |
| `avg_position_first_half` | mean GSC position, days 1-15, **zero-position rows excluded before averaging** | filled with column median, flagged via `has_position_data` (not filled with 0 - that would read as rank 1) | no | yes |
| `has_position_data` | whether any day 1-15 had real position data | n/a (it IS the missingness flag) | binary | yes |
| `active_days_first_half` | count of days 1-15 with impressions > 0 | none needed | no | yes |
| `pos_bucket_*` (one-hot) | which position band the item sits in, days 1-15 | inherits from avg_position handling above | yes, one-hot | yes |
| `volume_tier_*` (one-hot) | which impression-volume quartile, days 1-15 | none needed (quantile always defined) | yes, one-hot | yes |
| `client_hash_id`, `content_hash_id` | pseudonym IDs | n/a | n/a | context only - never fed to a model as a feature |


## 3. The leakage hunt

Attack the feature vector directly: (a) prove no feature is built from days 16-31, (b) prove no ID column leaked in as a feature, (c) show what a genuinely leaky feature does to a quick model, so the difference is visible rather than assumed.

In [3]:
# (a) Column-name / provenance check: every _fh / first_half feature must trace to days<=15 only.
feature_cols = [c for c in feature_vector.columns if c not in ('client_hash_id','content_hash_id')]
suspicious = [c for c in feature_cols if 'second' in c.lower() or 'sh' == c.lower()[-2:] and 'fh' not in c.lower()]
print('Feature columns:', feature_cols)
print('Any column name suggesting second-half/future data:', suspicious if suspicious else 'none')

# (b) ID-as-feature check: pseudonym IDs must never be model inputs.
id_cols_in_features = [c for c in feature_cols if 'hash_id' in c.lower()]
print('ID columns accidentally left in as features:', id_cols_in_features if id_cols_in_features else 'none')


Feature columns: ['clicks_first_half', 'impressions_first_half', 'ctr_first_half', 'avg_position_first_half', 'has_position_data', 'active_days_first_half', 'pos_bucket_1-3', 'pos_bucket_4-6', 'pos_bucket_7-10', 'pos_bucket_11-20', 'pos_bucket_21-50', 'pos_bucket_51+', 'volume_tier_Q1_low', 'volume_tier_Q2', 'volume_tier_Q3', 'volume_tier_Q4_high']
Any column name suggesting second-half/future data: none
ID columns accidentally left in as features: none


In [4]:
# (c) The attack: build the label, then show what an OBVIOUSLY leaky feature does vs the honest set.
label_df = (second_half.groupby(['client_hash_id','content_hash_id'])
            .agg(clicks_second_half=('gsc_clicks','sum')).reset_index())

scored = feature_vector.merge(label_df, on=['client_hash_id','content_hash_id'], how='inner')
scored['is_declining_label'] = (scored['clicks_second_half'] < scored['clicks_first_half']).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

honest_cols = [c for c in feature_cols]
df_model = scored.dropna(subset=honest_cols + ['is_declining_label'])

Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    df_model[honest_cols], df_model['is_declining_label'], test_size=0.25, random_state=42,
    stratify=df_model['is_declining_label'])
honest_model = LogisticRegression(max_iter=1000).fit(Xh_train, yh_train)
honest_acc = accuracy_score(yh_test, honest_model.predict(Xh_test))

leaky_cols = honest_cols + ['clicks_second_half']  # the attack: feed in the future window directly
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    df_model[leaky_cols], df_model['is_declining_label'], test_size=0.25, random_state=42,
    stratify=df_model['is_declining_label'])
leaky_model = LogisticRegression(max_iter=1000).fit(Xl_train, yl_train)
leaky_acc = accuracy_score(yl_test, leaky_model.predict(Xl_test))

print(f'honest feature vector accuracy:  {honest_acc:.3f}')
print(f'with clicks_second_half added:   {leaky_acc:.3f}  <-- inflated, confirms leakage, never ship this')


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


honest feature vector accuracy:  0.801
with clicks_second_half added:   1.000  <-- inflated, confirms leakage, never ship this


**Result:** `clicks_second_half` is excluded from the shipped feature vector. It was added here only to prove the leakage test catches it — the honest accuracy above is the one that ships.

## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `clicks_second_half` (and anything from days 16-31) | This is the future window the label is derived from - the exact leak demonstrated in section 3. |
| `client_hash_id`, `content_hash_id` | Pseudonym IDs - grouping/joining only, never a model feature (per `flyrank-data`). |
| `gsc_sum_position` | Redundant with `gsc_avg_position` (same underlying signal, just unnormalized by day count) - collinear, adds no information the average doesn't already carry. |
| `ga4_*` columns | Zero-filled before each client's `ga4_data_start` - unreliable for a large share of clients in this warehouse and not needed for a GSC-only baseline lane. |
| `sessions_*`, `ai_*` channel columns | Out of scope for this lane - these describe traffic-source mix, not GSC visibility/CTR, and pulling them in would need their own missingness/leakage review before use. |
| `report_date` (raw, as a feature) | Used only to build `day_of_month` for the first/second-half split - the raw date itself isn't informative once that split is made and would just re-encode row position in time. |
